# LeetCode 679 — 24 Game bằng Python

- Link: https://leetcode.com/problems/24-game/
- Độ khó: **Hard**
- Chủ đề: Backtracking (quay lui), Duyệt toàn bộ có cắt tỉa, Phân số chính xác (`Fraction`), Quy hoạch động trên tập con (bitmask)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def judgePoint24(self, cards: List[int]) -> bool:

```

## 1. Giới thiệu & mục tiêu học

Trò chơi 24 điểm rất quen thuộc: cầm 4 lá bài, dùng `+ - * /` và dấu ngoặc để tạo ra số 24. Bài LeetCode này hỏi: **với 4 lá cho trước, có cách nào tạo ra 24 không?**

Nghe như chỉ cần "thử hết", nhưng chính việc *thử hết cho đúng* mới là chỗ khó: thử những gì? thứ tự nào? dấu ngoặc đặt ở đâu? chia số thực thì so sánh với 24 thế nào cho khỏi sai số? Notebook này đi từ cách thử "ngây thơ" (liệt kê biểu thức) đến cách quay lui gọn gàng mà mọi người hay dùng, rồi đến cách quy hoạch động trên tập con.

Sau notebook này bạn sẽ:

1. Hiểu vì sao "chọn 2 số, gộp thành 1 số, lặp lại" bao trùm **mọi** cách đặt dấu ngoặc.
2. Viết được hàm quay lui (backtracking) trên danh sách số: chọn cặp → thử phép tính → đệ quy → quay lại.
3. Biết xử lý phép chia: so sánh số thực bằng `EPS`, hoặc dùng `Fraction` để tính **chính xác tuyệt đối**.
4. Biết cắt tỉa: phép `+` và `*` giao hoán nên chỉ cần thử một thứ tự; không chia cho 0.
5. Biết cách tiếp cận "tập giá trị của mỗi tập con" (DP trên bitmask) và kiểm chứng mọi cách với brute force.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `cards` gồm **đúng 4 số nguyên** trong đoạn `[1, 9]`. Hãy sắp xếp 4 số này thành một biểu thức toán học dùng các phép `+`, `-`, `*`, `/` và dấu ngoặc `(`, `)` sao cho giá trị bằng **24**. Trả về `True` nếu làm được, ngược lại `False`.

Các luật quan trọng:

- Phép chia `/` là **chia số thực**, không phải chia nguyên. Ví dụ `4 / (1 - 2/3) = 4 / (1/3) = 12`.
- Mọi phép toán đều là **phép hai ngôi** (giữa hai số). Đặc biệt **không** được dùng `-` như dấu âm một ngôi: với `[1, 1, 1, 1]`, biểu thức `-1 - 1 - 1 + 1` là **không hợp lệ**.
- **Không** được ghép chữ số: với `[1, 2, 1, 2]`, không được viết `12 + 12`.
- Mỗi lá phải dùng **đúng một lần**, nhưng được đổi thứ tự tuỳ ý.

**Ví dụ 1**

```text
Input:  cards = [4, 1, 8, 7]
Output: true
Giải thích: (8 - 4) * (7 - 1) = 4 * 6 = 24
```

**Ví dụ 2**

```text
Input:  cards = [1, 2, 1, 2]
Output: false
```

**Ràng buộc**

- `cards.length == 4`
- `1 <= cards[i] <= 9`

Ta chạy thử trước vài phép tính để làm quen.

In [1]:
print((8 - 4) * (7 - 1))      # ví dụ 1
print(4 / (1 - 2 / 3))        # chia số thực: ~12 (có thể lệch chút xíu!)
print(8 / (3 - 8 / 3))        # một lời giải nổi tiếng của [3, 3, 8, 8]
print(5 * (5 - 1 / 5))        # lời giải của [1, 5, 5, 5]

24
11.999999999999998
23.99999999999999
24.0


Để ý hai dòng cuối: kết quả có thể ra `23.999999999999996` thay vì `24.0`. Đây là **sai số dấu phẩy động** — ta sẽ phải xử lý nó cẩn thận (mục 3.3).

## 3. Kiến thức nền

### 3.1. `itertools.permutations` và `itertools.product`

- `permutations(xs)` sinh mọi **hoán vị** (thứ tự sắp xếp) của `xs`. Với 4 phần tử có `4! = 24` hoán vị (kể cả trùng nếu có số lặp lại).
- `product('+-*/', repeat=3)` sinh mọi bộ 3 phép toán: `4^3 = 64` bộ.

In [2]:
from itertools import permutations, product

perms = list(permutations([1, 2, 3]))
print(len(perms), perms)

ops3 = list(product('+-*/', repeat=3))
print(len(ops3), ops3[:6], '...')

6 [(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
64 [('+', '+', '+'), ('+', '+', '-'), ('+', '+', '*'), ('+', '+', '/'), ('+', '-', '+'), ('+', '-', '-')] ...


### 3.2. Đệ quy và quay lui (backtracking)

**Đệ quy** là hàm tự gọi lại chính nó trên bài toán nhỏ hơn. **Quay lui** là kiểu đệ quy "thử một lựa chọn → đi sâu → nếu không thành công thì quay lại thử lựa chọn khác".

Ví dụ nhỏ: liệt kê mọi cách chọn 2 phần tử *có thứ tự* từ danh sách và in ra.

In [3]:
def choose_pairs(xs):
    result = []
    for i in range(len(xs)):
        for j in range(len(xs)):
            if i != j:
                rest = [xs[t] for t in range(len(xs)) if t != i and t != j]
                result.append((xs[i], xs[j], rest))
    return result

for a, b, rest in choose_pairs([4, 1, 8]):
    print(f"chọn a={a}, b={b}, còn lại {rest}")

chọn a=4, b=1, còn lại [8]
chọn a=4, b=8, còn lại [1]
chọn a=1, b=4, còn lại [8]
chọn a=1, b=8, còn lại [4]
chọn a=8, b=4, còn lại [1]
chọn a=8, b=1, còn lại [4]


Mỗi dòng là một lựa chọn: lấy `a`, `b` ra, phần còn lại là `rest`. Lời giải quay lui của ta sẽ: gộp `a` và `b` thành **một** số mới `v`, rồi đệ quy trên `rest + [v]` (danh sách ngắn hơn 1 phần tử).

### 3.3. Sai số số thực và `EPS`

Máy tính lưu số thực ở dạng nhị phân nên `1/3` không lưu chính xác được. Do đó **không** so sánh `x == 24` với số thực, mà so `abs(x - 24) < EPS` với `EPS` nhỏ như `1e-6`.

In [4]:
x = 8 / (3 - 8 / 3)
print(repr(x), x == 24, abs(x - 24) < 1e-6)

23.99999999999999 False True


### 3.4. `fractions.Fraction` — phân số chính xác

`Fraction(a, b)` biểu diễn phân số `a/b` **chính xác tuyệt đối** (tử và mẫu là số nguyên Python, không giới hạn độ lớn). Cộng trừ nhân chia phân số cho ra phân số. Nhờ vậy `Fraction(8) / (3 - Fraction(8, 3)) == 24` là đúng hoàn toàn, khỏi lo `EPS`.

Nhược điểm: chậm hơn `float` vài lần. Với 4 lá bài thì không đáng kể.

In [5]:
from fractions import Fraction

y = Fraction(8) / (3 - Fraction(8) / 3)
print(y, y == 24)
print(Fraction(1, 3) + Fraction(1, 6))    # 1/2
try:
    Fraction(1) / 0
except ZeroDivisionError as e:
    print("Chia cho 0 ->", type(e).__name__)

24 True
1/2
Chia cho 0 -> ZeroDivisionError


### 3.5. Bitmask để biểu diễn tập con

Với 4 lá, mỗi **tập con** các lá có thể mã hoá bằng một số nguyên 4 bit: bit `i` bật nghĩa là có lá thứ `i`. Ví dụ `0b0101 = 5` là tập {lá 0, lá 2}. Có `2^4 = 16` tập con (kể cả rỗng).

Mẹo duyệt **mọi tập con con** `sub` của `mask`: `sub = (sub - 1) & mask`.

In [6]:
mask = 0b1011
sub = mask
subs = []
while sub:
    subs.append(format(sub, '04b'))
    sub = (sub - 1) & mask
print("các tập con khác rỗng của 1011:", subs)

các tập con khác rỗng của 1011: ['1011', '1010', '1001', '1000', '0011', '0010', '0001']


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Một biểu thức thực chất là một **cây**

Biểu thức `(8 - 4) * (7 - 1)` là một cây nhị phân: gốc là `*`, con trái là cây `8 - 4`, con phải là cây `7 - 1`. Lá cây là 4 lá bài. Mọi biểu thức hợp lệ (phép hai ngôi + ngoặc) đều là một cây như vậy, có đúng 4 lá và 3 nút phép toán.

### 4.2. Tính tay: "gộp dần" cây từ dưới lên

Tính một cây từ dưới lên giống như: **chọn hai số, thay bằng kết quả của chúng**, cho đến khi còn một số.

Với `[4, 1, 8, 7]` và biểu thức `(8 - 4) * (7 - 1)`:

| Bước | Danh sách hiện tại | Chọn | Phép | Kết quả mới |
|---|---|---|---|---|
| 0 | `[4, 1, 8, 7]` | 8, 4 | `8 - 4` | 4 |
| 1 | `[1, 7, 4]` | 7, 1 | `7 - 1` | 6 |
| 2 | `[4, 6]` | 4, 6 | `4 * 6` | 24 |
| 3 | `[24]` | — | — | **= 24 ✓** |

Đảo ngược quan sát: **nếu ta thử mọi cách "chọn 2 số + chọn phép" ở mỗi bước**, ta sẽ đi qua mọi cây biểu thức có thể. Đó là hạt giống của lời giải quay lui.

### 4.3. Có bao nhiêu biểu thức?

- Thứ tự 4 lá: `4! = 24`.
- Phép toán cho 3 vị trí: `4^3 = 64`.
- Cách đặt ngoặc (số cây nhị phân 4 lá — số Catalan `C_3`): `5`.

Tổng: `24 × 64 × 5 = 7680` biểu thức. Rất nhỏ! Nên bài này "Hard" không phải vì độ phức tạp, mà vì phải **liệt kê đúng và đủ**, và xử lý phép chia.

In [7]:
import math
print("số biểu thức tối đa:", math.factorial(4) * 4 ** 3 * 5)

# Thử tay vài lời giải với Fraction để thấy chúng đúng tuyệt đối
F = Fraction
print((F(8) - 4) * (F(7) - 1))
print(F(8) / (F(3) - F(8) / 3))
print(F(5) * (F(5) - F(1) / 5))

số biểu thức tối đa: 7680
24
24
24


## 5. Cách 1 — Brute Force: liệt kê mọi biểu thức (hoán vị × phép × ngoặc)

**Ý tưởng.** Đúng như phép đếm ở mục 4.3: với mỗi hoán vị `(a, b, c, d)`, mỗi bộ phép `(o1, o2, o3)` và mỗi trong 5 dạng ngoặc, tính giá trị bằng `Fraction`. Nếu có giá trị nào bằng 24 thì trả `True`.

Năm dạng ngoặc (5 cây nhị phân có 4 lá theo thứ tự `a b c d`):

1. `((a o1 b) o2 c) o3 d`
2. `(a o1 (b o2 c)) o3 d`
3. `(a o1 b) o2 (c o3 d)`
4. `a o1 ((b o2 c) o3 d)`
5. `a o1 (b o2 (c o3 d))`

Hàm `apply` trả `None` nếu gặp chia cho 0 hoặc nếu một vế đã là `None` (lan truyền "biểu thức không hợp lệ").

In [8]:
OPS = '+-*/'

def apply(x, op, y):
    """Tính x op y với Fraction; trả None nếu không hợp lệ (chia 0)."""
    if x is None or y is None:
        return None
    if op == '+':
        return x + y
    if op == '-':
        return x - y
    if op == '*':
        return x * y
    if y == 0:
        return None
    return x / y

SHAPES = [
    lambda a, b, c, d, o1, o2, o3: apply(apply(apply(a, o1, b), o2, c), o3, d),
    lambda a, b, c, d, o1, o2, o3: apply(apply(a, o1, apply(b, o2, c)), o3, d),
    lambda a, b, c, d, o1, o2, o3: apply(apply(a, o1, b), o2, apply(c, o3, d)),
    lambda a, b, c, d, o1, o2, o3: apply(a, o1, apply(apply(b, o2, c), o3, d)),
    lambda a, b, c, d, o1, o2, o3: apply(a, o1, apply(b, o2, apply(c, o3, d))),
]
SHAPE_FMT = [
    "(({a} {o1} {b}) {o2} {c}) {o3} {d}",
    "({a} {o1} ({b} {o2} {c})) {o3} {d}",
    "({a} {o1} {b}) {o2} ({c} {o3} {d})",
    "{a} {o1} (({b} {o2} {c}) {o3} {d})",
    "{a} {o1} ({b} {o2} ({c} {o3} {d}))",
]

def judge_brute(cards):
    for a, b, c, d in set(permutations([Fraction(x) for x in cards])):
        for o1, o2, o3 in product(OPS, repeat=3):
            for shape in SHAPES:
                if shape(a, b, c, d, o1, o2, o3) == 24:
                    return True
    return False

def find_expr_brute(cards):
    """Trả về một chuỗi biểu thức = 24 (hoặc None) — để người đọc kiểm tra bằng mắt."""
    for p in set(permutations(cards)):
        fp = [Fraction(x) for x in p]
        for o1, o2, o3 in product(OPS, repeat=3):
            for shape, fmt in zip(SHAPES, SHAPE_FMT):
                if shape(*fp, o1, o2, o3) == 24:
                    return fmt.format(a=p[0], b=p[1], c=p[2], d=p[3], o1=o1, o2=o2, o3=o3)
    return None

Ghi chú: `set(permutations(...))` bỏ các hoán vị trùng khi có lá bằng nhau (ví dụ `[3, 3, 8, 8]` chỉ có 6 hoán vị khác nhau thay vì 24). Không ảnh hưởng đúng sai, chỉ giảm việc thừa.

Chạy thử:

In [9]:
for cards in [[4, 1, 8, 7], [1, 2, 1, 2], [3, 3, 8, 8], [1, 5, 5, 5], [1, 1, 1, 1]]:
    print(cards, judge_brute(cards), find_expr_brute(cards))

[4, 1, 8, 7] True (1 * 8) * (7 - 4)
[1, 2, 1, 2] False None
[3, 3, 8, 8] True 8 / (3 - (8 / 3))
[1, 5, 5, 5] True 5 * (5 - (1 / 5))
[1, 1, 1, 1] False None


**Độ phức tạp.** Số biểu thức là hằng số (≤ 7680), mỗi biểu thức 3 phép tính → `O(1)` theo nghĩa đầu vào cố định 4 lá. Nhưng nếu tổng quát hoá cho `n` lá, số biểu thức là `n! · 4^(n-1) · Catalan(n-1)` — bùng nổ cực nhanh.

**Nút thắt.**

- Phải **liệt kê tay 5 dạng ngoặc** → khó tổng quát cho `n` lá, rất dễ quên một dạng.
- Rất nhiều việc thừa: `a + b` và `b + a` được thử cả hai; các hoán vị khác nhau sinh cùng một cây.
- Code dài, khó đọc.

Ta cần một cách "tự sinh" mọi cây mà không phải liệt kê dạng ngoặc.

## 6. Bước đột phá

**Insight:** Thay vì nghĩ "biểu thức hoàn chỉnh trông thế nào", hãy nghĩ "**phép tính đầu tiên** được thực hiện là gì".

Trong bất kỳ cây biểu thức nào, luôn có ít nhất một nút phép toán mà **cả hai con đều là lá** (một cặp số gốc). Thực hiện phép đó trước → ta được một danh sách **3 số** (2 số chưa dùng + kết quả). Bài toán với 3 số có cùng dạng với bài toán gốc! Lặp lại đến khi còn 1 số.

**Quy tắc "nếu… thì…":**

- **Nếu** danh sách chỉ còn 1 số → trả về (số đó == 24).
- **Nếu** còn ≥ 2 số → với mọi cặp `(i, j)` và mọi phép `op`: tạo danh sách mới = các số khác `i, j` + `[nums[i] op nums[j]]`; **nếu** đệ quy trên danh sách mới trả `True` **thì** trả `True` ngay.
- **Nếu** thử hết mà không được → `False`.

**Bất biến (invariant):** mỗi danh sách trong đệ quy luôn là "giá trị của một rừng cây con", mỗi số là giá trị của một biểu thức con dùng các lá rời nhau, và hợp các lá = cả 4 lá ban đầu.

**Vì sao không sót dấu ngoặc?** Dạng `(a o b) o (c o d)` được sinh ra bằng: bước 1 gộp `a, b`; bước 2 gộp `c, d`; bước 3 gộp hai kết quả. Mọi cây đều có thể "tháo" từng nút có hai con là lá như vậy → quay lui sinh đủ mọi cây.

**Cắt tỉa:**

- `a + b == b + a`, `a * b == b * a` → với cặp không thứ tự `{i, j}` chỉ cần thử `+`, `*` một lần; `-` và `/` thì thử cả `a - b`, `b - a`, `a / b`, `b / a`.
- Không chia cho 0.
- Dừng ngay khi tìm thấy (`return True` lan ngược lên).

## 7. Các cách tiếp theo

Ta sẽ xây 3 cách nữa (tổng cộng 4 cách):

| Cách | Ý tưởng chính |
|---|---|
| Cách 2 | Quay lui "chọn cặp có thứ tự", số thực + `EPS` (lời giải phổ biến nhất) |
| Cách 3 | Quay lui "chọn cặp không thứ tự" + cắt tỉa giao hoán + `Fraction` chính xác |
| Cách 4 | DP trên tập con: tập giá trị đạt được của mỗi tập con lá bài |

### 7.1. Cách 2 — Quay lui với số thực và `EPS`

**Giải thích.** Đúng như quy tắc ở mục 6. Ở mỗi tầng, duyệt mọi cặp **có thứ tự** `(i, j)` với `i != j` (nên `a - b` và `b - a` tự động được thử qua hai cặp `(i, j)` và `(j, i)`), thử 4 phép (bỏ phép chia nếu `b ≈ 0`).

In [10]:
EPS = 1e-6

def judge_backtrack_float(cards):
    def dfs(nums):
        if len(nums) == 1:
            return abs(nums[0] - 24) < EPS
        n = len(nums)
        for i in range(n):
            for j in range(n):
                if i == j:
                    continue
                rest = [nums[t] for t in range(n) if t != i and t != j]
                a, b = nums[i], nums[j]
                candidates = [a + b, a - b, a * b]
                if abs(b) > EPS:
                    candidates.append(a / b)
                for v in candidates:
                    if dfs(rest + [v]):
                        return True
        return False
    return dfs([float(c) for c in cards])

for cards in [[4, 1, 8, 7], [1, 2, 1, 2], [3, 3, 8, 8], [1, 5, 5, 5], [1, 1, 1, 1]]:
    print(cards, judge_backtrack_float(cards))

[4, 1, 8, 7] True
[1, 2, 1, 2] False
[3, 3, 8, 8] True
[1, 5, 5, 5] True
[1, 1, 1, 1] False


Giải thích từng phần:

- `if len(nums) == 1` — điểm dừng: còn một số, đó chính là giá trị cả biểu thức.
- `rest = [...]` — các số **không** được chọn, giữ nguyên.
- `candidates` — các giá trị có thể sinh từ `a, b`. Phép chia chỉ thêm khi `|b| > EPS` (tránh chia cho 0 hoặc số siêu nhỏ).
- `dfs(rest + [v])` — danh sách mới ngắn hơn 1 phần tử. Nếu nhánh này thành công thì `return True` ngay (không cần thử tiếp).

**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def judgePoint24(self, cards: List[int]) -> bool:
        EPS = 1e-6
        def dfs(nums):
            if len(nums) == 1:
                return abs(nums[0] - 24) < EPS
            n = len(nums)
            for i in range(n):
                for j in range(n):
                    if i == j:
                        continue
                    rest = [nums[t] for t in range(n) if t != i and t != j]
                    a, b = nums[i], nums[j]
                    cand = [a + b, a - b, a * b]
                    if abs(b) > EPS:
                        cand.append(a / b)
                    for v in cand:
                        if dfs(rest + [v]):
                            return True
            return False
        return dfs([float(c) for c in cards])
```

**Độ phức tạp.** Tầng 4 số: 12 cặp × 4 phép = 48 nhánh; tầng 3 số: 6 × 4 = 24; tầng 2 số: 2 × 4 = 8. Tổng lá tối đa `48 × 24 × 8 = 9216` — hằng số, `O(1)` time, `O(1)` space (độ sâu đệ quy 3).

**Vì sao `EPS = 1e-6` an toàn?** Các giá trị trung gian là phân số có tử, mẫu nhỏ (mẫu ≤ vài trăm). Nếu một giá trị ≠ 24 thì nó cách 24 ít nhất `1/mẫu`, lớn hơn rất nhiều so với `1e-6`; còn sai số float chỉ cỡ `1e-14`. Nên không có nhầm lẫn.

### 7.2. Cách 3 — Quay lui cặp không thứ tự + cắt tỉa + `Fraction`

**Giải thích.** Chỉ duyệt `i < j` (cặp không thứ tự). Với cặp `(a, b)` sinh 6 giá trị: `a+b`, `a*b`, `a-b`, `b-a`, `a/b` (nếu `b ≠ 0`), `b/a` (nếu `a ≠ 0`). Dùng `Fraction` nên so sánh `== 24` chính xác tuyệt đối — không còn câu hỏi "EPS bao nhiêu là đủ".

Thêm một cắt tỉa nhỏ: dùng `set` cho các ứng viên để khỏi đệ quy hai lần cùng một giá trị (ví dụ `a = b` thì `a - b == b - a`).

In [11]:
def combine(a, b):
    """Mọi giá trị sinh được từ cặp không thứ tự {a, b} (Fraction)."""
    out = {a + b, a * b, a - b, b - a}
    if b != 0:
        out.add(a / b)
    if a != 0:
        out.add(b / a)
    return out

def judge_backtrack_frac(cards):
    def dfs(nums):
        if len(nums) == 1:
            return nums[0] == 24
        n = len(nums)
        for i in range(n):
            for j in range(i + 1, n):
                rest = [nums[t] for t in range(n) if t != i and t != j]
                for v in combine(nums[i], nums[j]):
                    if dfs(rest + [v]):
                        return True
        return False
    return dfs([Fraction(c) for c in cards])

print(combine(Fraction(3), Fraction(8, 3)))
for cards in [[4, 1, 8, 7], [1, 2, 1, 2], [3, 3, 8, 8], [1, 5, 5, 5], [1, 1, 1, 1]]:
    print(cards, judge_backtrack_frac(cards))

{Fraction(9, 8), Fraction(8, 1), Fraction(-1, 3), Fraction(8, 9), Fraction(17, 3), Fraction(1, 3)}
[4, 1, 8, 7] True
[1, 2, 1, 2] False
[3, 3, 8, 8] True
[1, 5, 5, 5] True
[1, 1, 1, 1] False


**Code nộp LeetCode (Cách 3):**

```python
from fractions import Fraction

class Solution:
    def judgePoint24(self, cards: List[int]) -> bool:
        def combine(a, b):
            out = {a + b, a * b, a - b, b - a}
            if b != 0: out.add(a / b)
            if a != 0: out.add(b / a)
            return out
        def dfs(nums):
            if len(nums) == 1:
                return nums[0] == 24
            n = len(nums)
            for i in range(n):
                for j in range(i + 1, n):
                    rest = [nums[t] for t in range(n) if t != i and t != j]
                    for v in combine(nums[i], nums[j]):
                        if dfs(rest + [v]):
                            return True
            return False
        return dfs([Fraction(c) for c in cards])
```

**Độ phức tạp.** Tầng 4: 6 cặp × ≤ 6 giá trị = 36; tầng 3: 3 × 6 = 18; tầng 2: 1 × 6 = 6. Tổng ≤ `36 × 18 × 6 = 3888` lá — ít hơn Cách 2 khoảng 2,4 lần, bù lại mỗi phép `Fraction` chậm hơn `float`. Vẫn `O(1)`.

Ta đếm thực tế số lần gọi đệ quy để thấy hiệu quả cắt tỉa:

In [12]:
def count_calls(cards, mode):
    calls = [0]
    def dfs_f(nums):
        calls[0] += 1
        if len(nums) == 1:
            return False          # cố ý không dừng sớm để đếm toàn bộ cây
        n = len(nums)
        for i in range(n):
            for j in range(n):
                if i != j:
                    rest = [nums[t] for t in range(n) if t != i and t != j]
                    a, b = nums[i], nums[j]
                    cand = [a + b, a - b, a * b] + ([a / b] if abs(b) > EPS else [])
                    for v in cand:
                        dfs_f(rest + [v])
        return False
    def dfs_q(nums):
        calls[0] += 1
        if len(nums) == 1:
            return False
        n = len(nums)
        for i in range(n):
            for j in range(i + 1, n):
                rest = [nums[t] for t in range(n) if t != i and t != j]
                for v in combine(nums[i], nums[j]):
                    dfs_q(rest + [v])
        return False
    if mode == 'float':
        dfs_f([float(c) for c in cards])
    else:
        dfs_q([Fraction(c) for c in cards])
    return calls[0]

print("Cách 2 (cặp có thứ tự):  ", count_calls([1, 2, 1, 2], 'float'), "lần gọi")
print("Cách 3 (cặp không thứ tự):", count_calls([1, 2, 1, 2], 'frac'), "lần gọi")

Cách 2 (cặp có thứ tự):   10201 lần gọi
Cách 3 (cặp không thứ tự): 1911 lần gọi


### 7.3. Cách 4 — DP trên tập con (tập giá trị của mỗi tập con)

**Giải thích.** Nhìn từ trên xuống: gốc cây chia 4 lá thành **hai nhóm** (tập con `A` và phần bù `B`), mỗi nhóm tự tạo thành một biểu thức con. Vậy:

> `vals(mask)` = tập mọi giá trị biểu thức tạo được từ đúng các lá trong `mask`.
>
> - Nếu `mask` chỉ có 1 lá `i`: `vals(mask) = {cards[i]}`.
> - Ngược lại: `vals(mask) = ⋃` trên mọi cách chia `mask = A ⊔ B` (A, B khác rỗng) của `{x op y : x ∈ vals(A), y ∈ vals(B)}`.

Đáp án: `24 ∈ vals(1111)`.

Vì `x op y` với `op` ∈ {`-`, `/`} không giao hoán, ta duyệt cả `(A, B)` lẫn `(B, A)` — điều này tự xảy ra khi duyệt mọi tập con `A` của `mask`.

Ưu điểm: tính một lần cho mỗi tập con (memo), dùng lại nhiều lần; cho biết **mọi** giá trị đạt được, không chỉ 24 (rất hợp cho biến thể "đạt được những số nào?").

In [13]:
from functools import lru_cache

def all_values(cards):
    n = len(cards)
    fc = [Fraction(c) for c in cards]

    @lru_cache(maxsize=None)
    def vals(mask):
        if mask & (mask - 1) == 0:              # chỉ một bit bật
            return frozenset([fc[mask.bit_length() - 1]])
        out = set()
        sub = (mask - 1) & mask
        while sub:
            other = mask ^ sub
            for x in vals(sub):
                for y in vals(other):
                    out.add(x + y)
                    out.add(x - y)
                    out.add(x * y)
                    if y != 0:
                        out.add(x / y)
            sub = (sub - 1) & mask
        return frozenset(out)

    return vals((1 << n) - 1)

def judge_dp(cards):
    return Fraction(24) in all_values(cards)

for cards in [[4, 1, 8, 7], [1, 2, 1, 2], [3, 3, 8, 8], [1, 5, 5, 5], [1, 1, 1, 1]]:
    print(cards, judge_dp(cards), "số giá trị đạt được:", len(all_values(cards)))

[4, 1, 8, 7] True số giá trị đạt được: 526
[1, 2, 1, 2] False số giá trị đạt được: 38
[3, 3, 8, 8] True số giá trị đạt được: 269
[1, 5, 5, 5] True số giá trị đạt được: 125
[1, 1, 1, 1] False số giá trị đạt được: 11


Giải thích:

- `mask & (mask - 1) == 0` — mẹo kiểm tra "chỉ có một bit": xoá bit thấp nhất mà còn 0 thì ban đầu chỉ có 1 bit.
- `mask.bit_length() - 1` — vị trí bit duy nhất đó (chỉ số lá).
- Vòng `while sub` duyệt mọi tập con thật sự khác rỗng `sub ⊂ mask` (bắt đầu từ `(mask-1) & mask` để bỏ chính `mask`), `other = mask ^ sub` là phần bù.
- `lru_cache` ghi nhớ kết quả mỗi `mask` (tham số phải hashable — `int` thì ổn).

**Code nộp LeetCode (Cách 4):**

```python
from fractions import Fraction
from functools import lru_cache

class Solution:
    def judgePoint24(self, cards: List[int]) -> bool:
        fc = [Fraction(c) for c in cards]
        @lru_cache(maxsize=None)
        def vals(mask):
            if mask & (mask - 1) == 0:
                return frozenset([fc[mask.bit_length() - 1]])
            out = set()
            sub = (mask - 1) & mask
            while sub:
                other = mask ^ sub
                for x in vals(sub):
                    for y in vals(other):
                        out.add(x + y); out.add(x - y); out.add(x * y)
                        if y != 0:
                            out.add(x / y)
                sub = (sub - 1) & mask
            return frozenset(out)
        return Fraction(24) in vals(15)
```

**Độ phức tạp.** Với `n` lá: có `2^n` mask, mỗi mask duyệt các tập con → tổng `3^n` cặp (A, B); nhưng kích thước tập giá trị tăng rất nhanh theo `n`. Với `n = 4` mọi thứ là hằng số nhỏ. Không dừng sớm được như quay lui (phải tính trọn tập), nên thường chậm hơn Cách 2/3 khi đáp án là `True`.

## 8. TRACE từng bước

### 8.1. Bảng trạng thái (tính tay) cho `[3, 3, 8, 8]`

Lời giải `8 / (3 - 8/3)`. Trong ngôn ngữ "gộp cặp":

| Tầng | Danh sách (Fraction) | Cặp chọn | Phép | Giá trị mới |
|---|---|---|---|---|
| 0 | `[3, 3, 8, 8]` | 8, 3 | `8 / 3` | `8/3` |
| 1 | `[3, 8, 8/3]` | 3, 8/3 | `3 - 8/3` | `1/3` |
| 2 | `[8, 1/3]` | 8, 1/3 | `8 / (1/3)` | `24` |
| 3 | `[24]` | — | — | ✓ |

Chú ý: đây là ví dụ kinh điển **không thể** giải nếu chỉ dùng số nguyên — phải đi qua phân số `8/3` và `1/3`.

### 8.2. Hàm `trace_backtrack` — in đường đi thành công

Hàm dưới đây giống Cách 3 nhưng mang theo **chuỗi biểu thức** của mỗi số, và in danh sách ở mỗi tầng trên đường đi dẫn tới 24.

In [14]:
def trace_backtrack(cards, verbose=True):
    """Quay lui Fraction; mỗi phần tử là (giá trị, chuỗi biểu thức). Trả về chuỗi biểu thức hoặc None."""
    path = []          # lưu trạng thái các tầng trên nhánh hiện tại
    stats = {'calls': 0}

    def options(a, b):
        (va, ea), (vb, eb) = a, b
        res = [(va + vb, f"({ea} + {eb})"), (va * vb, f"({ea} * {eb})"),
               (va - vb, f"({ea} - {eb})"), (vb - va, f"({eb} - {ea})")]
        if vb != 0:
            res.append((va / vb, f"({ea} / {eb})"))
        if va != 0:
            res.append((vb / va, f"({eb} / {ea})"))
        return res

    def dfs(items):
        stats['calls'] += 1
        path.append(items)
        if len(items) == 1:
            if items[0][0] == 24:
                return items[0][1]
            path.pop()
            return None
        n = len(items)
        for i in range(n):
            for j in range(i + 1, n):
                rest = [items[t] for t in range(n) if t != i and t != j]
                for v, e in options(items[i], items[j]):
                    got = dfs(rest + [(v, e)])
                    if got:
                        return got
        path.pop()
        return None

    ans = dfs([(Fraction(c), str(c)) for c in cards])
    if verbose:
        print(f"cards={cards}  -> {'TÌM THẤY' if ans else 'KHÔNG CÓ'}  (số lần gọi dfs: {stats['calls']})")
        if ans:
            for depth, items in enumerate(path):
                vals = ', '.join(str(v) for v, _ in items)
                print(f"  tầng {depth}: [{vals}]")
            print("  biểu thức:", ans)
    return ans

trace_backtrack([3, 3, 8, 8])
trace_backtrack([4, 1, 8, 7])
trace_backtrack([1, 2, 1, 2])

cards=[3, 3, 8, 8]  -> TÌM THẤY  (số lần gọi dfs: 1401)
  tầng 0: [3, 3, 8, 8]
  tầng 1: [3, 8, 8/3]
  tầng 2: [8, 1/3]
  tầng 3: [24]
  biểu thức: (8 / (3 - (8 / 3)))
cards=[4, 1, 8, 7]  -> TÌM THẤY  (số lần gọi dfs: 230)
  tầng 0: [4, 1, 8, 7]
  tầng 1: [8, 7, 4]
  tầng 2: [8, 3]
  tầng 3: [24]
  biểu thức: (8 * (7 - (4 * 1)))
cards=[1, 2, 1, 2]  -> KHÔNG CÓ  (số lần gọi dfs: 4413)


Đọc output:

- Với `[3, 3, 8, 8]` danh sách đi qua `8/3` rồi `1/3` đúng như bảng tay.
- Với `[1, 2, 1, 2]` không tìm thấy, và số lần gọi `dfs` chính là kích thước toàn bộ cây tìm kiếm (đã giảm bởi `set` trùng giá trị ở Cách 3 thì còn ít hơn nữa).

### 8.3. Trace DP: tập giá trị theo từng tập con

In kích thước `vals(mask)` cho mọi mask của `[4, 1, 8, 7]`, và vài giá trị mẫu.

In [15]:
def trace_dp(cards):
    fc = [Fraction(c) for c in cards]
    memo = {}
    def vals(mask):
        if mask in memo:
            return memo[mask]
        if mask & (mask - 1) == 0:
            memo[mask] = {fc[mask.bit_length() - 1]}
            return memo[mask]
        out = set()
        sub = (mask - 1) & mask
        while sub:
            for x in vals(sub):
                for y in vals(mask ^ sub):
                    out |= {x + y, x - y, x * y}
                    if y != 0:
                        out.add(x / y)
            sub = (sub - 1) & mask
        memo[mask] = out
        return out
    full = (1 << len(cards)) - 1
    vals(full)
    for mask in sorted(memo, key=lambda m: (bin(m).count('1'), m)):
        members = [cards[i] for i in range(len(cards)) if mask >> i & 1]
        sample = sorted(memo[mask])[:5]
        print(f"mask={mask:04b} lá={members!s:<14} |vals|={len(memo[mask]):>4}  vài giá trị: {[str(s) for s in sample]}")
    print("24 có trong vals(1111)?", Fraction(24) in memo[full])

trace_dp([4, 1, 8, 7])

mask=0001 lá=[4]            |vals|=   1  vài giá trị: ['4']
mask=0010 lá=[1]            |vals|=   1  vài giá trị: ['1']
mask=0100 lá=[8]            |vals|=   1  vài giá trị: ['8']
mask=1000 lá=[7]            |vals|=   1  vài giá trị: ['7']
mask=0011 lá=[4, 1]         |vals|=   5  vài giá trị: ['-3', '1/4', '3', '4', '5']
mask=0101 lá=[4, 8]         |vals|=   6  vài giá trị: ['-4', '1/2', '2', '4', '12']
mask=0110 lá=[1, 8]         |vals|=   5  vài giá trị: ['-7', '1/8', '7', '8', '9']
mask=1001 lá=[4, 7]         |vals|=   6  vài giá trị: ['-3', '4/7', '7/4', '3', '11']
mask=1010 lá=[1, 7]         |vals|=   5  vài giá trị: ['-6', '1/7', '6', '7', '8']
mask=1100 lá=[8, 7]         |vals|=   6  vài giá trị: ['-1', '7/8', '1', '8/7', '15']
mask=0111 lá=[4, 1, 8]      |vals|=  48  vài giá trị: ['-31', '-28', '-24', '-11', '-31/4']
mask=1011 lá=[4, 1, 7]      |vals|=  47  vài giá trị: ['-27', '-24', '-21', '-10', '-27/4']
mask=1101 lá=[4, 8, 7]      |vals|=  63  vài giá trị: ['-52', '-28', '-

## 9. Code nộp LeetCode

Chọn **Cách 3** (quay lui + `Fraction`): chính xác tuyệt đối, ngắn, nhanh. (Nếu thích số thực, Cách 2 cũng được chấp nhận.)

In [16]:
from typing import List
from fractions import Fraction

class Solution:
    def judgePoint24(self, cards: List[int]) -> bool:
        def combine(a, b):
            out = {a + b, a * b, a - b, b - a}
            if b != 0:
                out.add(a / b)
            if a != 0:
                out.add(b / a)
            return out

        def dfs(nums):
            if len(nums) == 1:
                return nums[0] == 24
            n = len(nums)
            for i in range(n):
                for j in range(i + 1, n):
                    rest = [nums[t] for t in range(n) if t != i and t != j]
                    for v in combine(nums[i], nums[j]):
                        if dfs(rest + [v]):
                            return True
            return False

        return dfs([Fraction(c) for c in cards])

sol = Solution()
assert sol.judgePoint24([4, 1, 8, 7]) is True
assert sol.judgePoint24([1, 2, 1, 2]) is False
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Đủ (không sót lời giải).** Giả sử có biểu thức hợp lệ bằng 24; đó là một cây nhị phân với 4 lá. Một cây có ≥ 2 lá luôn có một nút trong mà hai con đều là lá (nút trong sâu nhất). Gọi hai lá đó là `a, b` và phép là `op`. Thay nút này bằng một lá mới có giá trị `a op b` → cây mới có 3 lá, **cùng giá trị gốc**. Quay lui của ta, tại tầng hiện tại, có thử đúng cặp `{a, b}` với đúng giá trị `a op b` (vì `combine` chứa mọi `a op b` và `b op a`). Quy nạp theo số lá → quay lui sẽ đi tới danh sách `[24]`.

**Đúng (không báo nhầm).** Mỗi bước gộp tương ứng với việc tạo một nút phép toán hợp lệ (không chia 0). Nên mỗi khi đạt `[24]`, ta có thể dựng lại một cây biểu thức hợp lệ bằng 24 (chính là chuỗi biểu thức ở `trace_backtrack`).

**Không có toán tử một ngôi / ghép chữ số.** Ta chỉ sinh `+ - * /` giữa hai giá trị — đúng với luật đề.

**Chính xác số học.** Với `Fraction`, mọi phép là chính xác nên `== 24` không bao giờ sai. Với `float + EPS`, lập luận ở mục 7.1: giá trị khác 24 cách 24 xa hơn rất nhiều so với sai số làm tròn.

**DP (Cách 4)** đúng vì gốc của mọi cây chia các lá thành hai nhóm rời nhau, khác rỗng; quy nạp theo kích thước tập con.

## 11. Phân tích Time/Space

Với đầu vào cố định 4 lá, mọi cách đều `O(1)`. Ý nghĩa thực sự nằm ở **hằng số** và ở **khả năng tổng quát cho `n` lá**:

| Cách | Số trạng thái/biểu thức (n = 4) | Tổng quát n lá | Space |
|---|---|---|---|
| 1. Liệt kê biểu thức | ≤ 7680 biểu thức × 3 phép | `n!·4^(n-1)·C(n-1)` | `O(n)` |
| 2. Quay lui cặp có thứ tự | ≤ 9216 lá | `∏_{k=2..n} k(k-1)·4` | `O(n^2)` (danh sách mỗi tầng) |
| 3. Quay lui cặp không thứ tự + Fraction | ≤ 3888 lá | `∏_{k=2..n} C(k,2)·6` | `O(n^2)` |
| 4. DP tập con | 15 mask, tập giá trị vài trăm | `3^n` cặp mask × \|vals\|² | lưu mọi tập giá trị |

Ta đo thời gian thực tế trên toàn bộ 495 bộ bài khác nhau (bộ không tính thứ tự) của các số 1–9:

In [17]:
import time
from itertools import combinations_with_replacement

all_hands = [list(h) for h in combinations_with_replacement(range(1, 10), 4)]
print("số bộ bài khác nhau:", len(all_hands))

for name, fn in [("Cách 2 float", judge_backtrack_float),
                 ("Cách 3 Fraction", judge_backtrack_frac)]:
    t0 = time.perf_counter()
    cnt = sum(fn(h) for h in all_hands)
    print(f"{name:<16}: {cnt} bộ giải được, {time.perf_counter() - t0:.2f}s")

số bộ bài khác nhau: 495


Cách 2 float    : 404 bộ giải được, 0.24s


Cách 3 Fraction : 404 bộ giải được, 0.42s


## 12. Bộ test

Gồm:

1. Ví dụ đề.
2. Các trường hợp đặc biệt: cần phân số (`[3,3,8,8]`, `[1,5,5,5]`), tất cả bằng nhau, toàn số lớn, chứa 1.
3. Đối chiếu **mọi** cách với brute force trên các bộ ngẫu nhiên có seed.
4. Đối chiếu Cách 2/3/4 với nhau trên **toàn bộ** 495 bộ bài, và kiểm tra rằng mọi hoán vị của một bộ cho cùng kết quả.

In [18]:
import random

def check_all(cards, expected=None):
    r = [judge_brute(cards), judge_backtrack_float(cards), judge_backtrack_frac(cards),
         judge_dp(cards), Solution().judgePoint24(cards)]
    assert len(set(r)) == 1, (cards, r)
    if expected is not None:
        assert r[0] == expected, (cards, r[0], expected)
    return r[0]

# 1. Ví dụ đề
check_all([4, 1, 8, 7], True)
check_all([1, 2, 1, 2], False)

# 2. Edge case
check_all([3, 3, 8, 8], True)     # 8 / (3 - 8/3)
check_all([1, 5, 5, 5], True)     # 5 * (5 - 1/5)
check_all([1, 1, 1, 1], False)    # tối đa 4
check_all([6, 6, 6, 6], True)     # 6 + 6 + 6 + 6
check_all([9, 9, 9, 9], False)
check_all([1, 1, 1, 8], True)     # (1 + 1 + 1) * 8
check_all([1, 1, 7, 7], False)
check_all([1, 3, 4, 6], True)     # 6 / (1 - 3/4)
assert find_expr_brute([1, 3, 4, 6]) is not None
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


In [19]:
# 3. Ngẫu nhiên có seed: đối chiếu với brute force
random.seed(679)
for _ in range(60):
    cards = [random.randint(1, 9) for _ in range(4)]
    check_all(cards)
print("60 bộ ngẫu nhiên: các cách khớp brute force")

# 4. Toàn bộ 495 bộ: Cách 2, 3, 4 khớp nhau; kết quả không phụ thuộc thứ tự lá
random.seed(24)
solvable = 0
for h in all_hands:
    a = judge_backtrack_frac(h)
    assert a == judge_backtrack_float(h) == judge_dp(h), h
    shuffled = h[:]
    random.shuffle(shuffled)
    assert judge_backtrack_frac(shuffled) == a
    solvable += a
print(f"495 bộ: các cách khớp nhau; {solvable} bộ giải được, {495 - solvable} bộ không")

60 bộ ngẫu nhiên: các cách khớp brute force


495 bộ: các cách khớp nhau; 404 bộ giải được, 91 bộ không


## 13. Lỗi người mới hay gặp

### Lỗi 1: So sánh số thực bằng `==`

In [20]:
def judge_wrong_eq(cards):
    def dfs(nums):
        if len(nums) == 1:
            return nums[0] == 24              # SAI: float hiếm khi đúng bằng 24
        n = len(nums)
        for i in range(n):
            for j in range(n):
                if i != j:
                    rest = [nums[t] for t in range(n) if t != i and t != j]
                    a, b = nums[i], nums[j]
                    for v in [a + b, a - b, a * b] + ([a / b] if b else []):
                        if dfs(rest + [v]):
                            return True
        return False
    return dfs([float(c) for c in cards])

print("[3,3,8,8] sai bằng ==:", judge_wrong_eq([3, 3, 8, 8]), "| đúng:", judge_backtrack_float([3, 3, 8, 8]))

[3,3,8,8] sai bằng ==: False | đúng: True


**Vì sao sai:** `8 / (3 - 8/3)` cho `23.999999999999996`. Phải dùng `abs(x - 24) < EPS` hoặc `Fraction`.

### Lỗi 2: Chia nguyên `//` thay cho chia thực

In [21]:
def judge_wrong_intdiv(cards):
    def dfs(nums):
        if len(nums) == 1:
            return nums[0] == 24
        n = len(nums)
        for i in range(n):
            for j in range(n):
                if i != j:
                    rest = [nums[t] for t in range(n) if t != i and t != j]
                    a, b = nums[i], nums[j]
                    for v in [a + b, a - b, a * b] + ([a // b] if b else []):   # SAI: //
                        if dfs(rest + [v]):
                            return True
        return False
    return dfs(list(cards))

print("[1,5,5,5] sai với //:", judge_wrong_intdiv([1, 5, 5, 5]), "| đúng:", judge_backtrack_frac([1, 5, 5, 5]))

[1,5,5,5] sai với //: False | đúng: True


**Vì sao sai:** đề nói rõ chia số thực. `1 // 5 = 0` làm mất lời giải `5 * (5 - 1/5)`. Ngược lại `//` còn có thể tạo lời giải *giả* (ví dụ `7 // 2 = 3`).

### Lỗi 3: Chỉ thử `a - b`, `a / b` với `i < j` (quên `b - a`, `b / a`)

In [22]:
def judge_wrong_noncomm(cards):
    def dfs(nums):
        if len(nums) == 1:
            return nums[0] == 24
        n = len(nums)
        for i in range(n):
            for j in range(i + 1, n):                       # chỉ i < j ...
                rest = [nums[t] for t in range(n) if t != i and t != j]
                a, b = nums[i], nums[j]
                cand = {a + b, a * b, a - b}                # ... nhưng quên b - a, b / a
                if b != 0:
                    cand.add(a / b)
                for v in cand:
                    if dfs(rest + [v]):
                        return True
        return False
    return dfs([Fraction(c) for c in cards])

misses = [h for h in all_hands if judge_wrong_noncomm(h) != judge_backtrack_frac(h)]
print("số bộ bị trả lời sai:", len(misses), "ví dụ:", misses[:5])

số bộ bị trả lời sai: 33 ví dụ: [[1, 1, 2, 9], [1, 2, 2, 7], [1, 2, 4, 6], [1, 2, 7, 7], [1, 3, 5, 7]]


**Vì sao sai:** `-` và `/` **không giao hoán**. Khi chỉ duyệt cặp không thứ tự, phải tự thêm cả hai chiều. Code đúng: hàm `combine` ở Cách 3.

### Lỗi 4: Chỉ ghép tuần tự từ trái sang phải (quên dạng `(a o b) o (c o d)`)

Một số bạn viết "lấy số đầu, lần lượt ghép với số tiếp theo" theo mọi hoán vị. Cách này chỉ sinh dạng `((a o b) o c) o d` và mất `(8 - 4) * (7 - 1)`-kiểu khi không có cách tuần tự nào khác.

In [23]:
def judge_wrong_linear(cards):
    for p in permutations([Fraction(c) for c in cards]):
        for o1, o2, o3 in product(OPS, repeat=3):
            if apply(apply(apply(p[0], o1, p[1]), o2, p[2]), o3, p[3]) == 24:
                return True
    return False

misses = [h for h in all_hands if judge_wrong_linear(h) != judge_backtrack_frac(h)]
print("số bộ bị bỏ sót:", len(misses), "ví dụ:", misses[:6])
print("một lời giải của", misses[0], ":", find_expr_brute(misses[0]))

số bộ bị bỏ sót: 39 ví dụ: [[1, 1, 2, 7], [1, 1, 2, 9], [1, 1, 3, 5], [1, 1, 4, 9], [1, 1, 5, 7], [1, 1, 6, 8]]
một lời giải của [1, 1, 2, 7] : (1 + 7) * (1 + 2)


**Code đúng:** quay lui "chọn **bất kỳ** cặp nào trong danh sách hiện tại" (không chỉ cặp đầu) — kết quả mới được đưa vào cuối danh sách và có thể được ghép với bất kỳ số nào sau đó.

## 14. Mẹo & nhận dạng pattern

- **"Dùng mỗi phần tử đúng một lần, kết hợp hai cái một"** → quay lui "chọn cặp, gộp, đệ quy trên danh sách ngắn hơn". Giống: Burst Balloons (theo khoảng), Merge Stones, "Expression Add Operators".
- **"Mọi cách đặt ngoặc"** → nghĩ tới **chia đôi tại phép toán gốc** (DP khoảng / DP tập con), như LeetCode 241 *Different Ways to Add Parentheses*.
- **Đầu vào rất nhỏ** (4 phần tử, `n ≤ 15`…) là tín hiệu: thử hết được, chỉ cần thử **đúng và đủ**.
- **Có phép chia số thực** → nghĩ ngay tới `EPS` hoặc `Fraction`.
- **Cắt tỉa giao hoán**: với phép giao hoán, chỉ duyệt cặp `i < j`; với phép không giao hoán, thêm chiều ngược lại.
- Viết thêm hàm **trả về chuỗi biểu thức** để tự kiểm tra bằng mắt — rất hữu ích khi debug.

## 15. Biến thể

1. **In ra một biểu thức** (không chỉ True/False) → `trace_backtrack` ở mục 8.2.
2. **Đích tổng quát `target`**, `n` lá tuỳ ý → thay 24 bằng `target`; quay lui vẫn đúng nhưng bùng nổ theo `n`.
3. **Liệt kê mọi giá trị đạt được** → DP tập con (Cách 4).
4. **Chỉ cho phép số nguyên ở mọi bước** (chia phải chia hết) → trong `combine` chỉ thêm `a / b` khi `a % b == 0`.
5. **Đếm số bộ bài giải được** trong 1–13 (bài toán dân gian) → chạy hàm trên mọi `combinations_with_replacement`.

Ví dụ biến thể 2 + 4: đích tuỳ ý và chỉ chia hết.

In [24]:
def can_make(cards, target, integer_only=False):
    def comb(a, b):
        out = {a + b, a * b, a - b, b - a}
        if b != 0 and (not integer_only or a % b == 0):
            out.add(a / b if not integer_only else Fraction(a // b))
        if a != 0 and (not integer_only or b % a == 0):
            out.add(b / a if not integer_only else Fraction(b // a))
        return out
    def dfs(nums):
        if len(nums) == 1:
            return nums[0] == target
        n = len(nums)
        for i in range(n):
            for j in range(i + 1, n):
                rest = [nums[t] for t in range(n) if t != i and t != j]
                if any(dfs(rest + [v]) for v in comb(nums[i], nums[j])):
                    return True
        return False
    return dfs([Fraction(c) for c in cards])

print("[3,3,8,8] -> 24, chia thực:", can_make([3, 3, 8, 8], 24))
print("[3,3,8,8] -> 24, chỉ số nguyên:", can_make([3, 3, 8, 8], 24, integer_only=True))
print("[1,2,3,4,5] -> 100:", can_make([1, 2, 3, 4, 5], 100))
print("[2,3,7] -> 1:", can_make([2, 3, 7], 1))

[3,3,8,8] -> 24, chia thực: True
[3,3,8,8] -> 24, chỉ số nguyên: False
[1,2,3,4,5] -> 100: True
[2,3,7] -> 1: True


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time (n = 4) | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê biểu thức | Hoán vị × 64 bộ phép × 5 dạng ngoặc | ≤ 7680 biểu thức | `O(1)` | Làm brute force đối chiếu; khi muốn in đủ mọi biểu thức |
| 2. Quay lui float | Chọn cặp có thứ tự, gộp, đệ quy; so sánh bằng `EPS` | ≤ 9216 lá | `O(1)` | Lời giải phỏng vấn phổ biến, nhanh, ngắn |
| 3. Quay lui Fraction | Cặp không thứ tự + thêm `b-a`, `b/a`; chính xác | ≤ 3888 lá | `O(1)` | Khi muốn đúng tuyệt đối, không lo EPS (khuyên dùng) |
| 4. DP tập con | `vals(mask)` = hợp các `x op y` trên mọi cách chia mask | 15 mask, vài nghìn phép | lưu mọi tập giá trị | Khi cần **mọi** giá trị đạt được hoặc nhiều truy vấn đích |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code, đoán `judgePoint24` cho: `[1, 1, 1, 1]`, `[2, 2, 2, 3]`, `[1, 1, 2, 6]`.

**Bài 2 (tính tay).** Lập bảng "gộp cặp" như mục 8.1 cho lời giải của `[1, 3, 4, 6]` (gợi ý: có phép chia ra phân số).

**Bài 3 (điền TODO).** Hoàn thiện hàm `combine_todo(a, b)` trả về mọi giá trị từ cặp không thứ tự `{a, b}`.

**Bài 4 (tự viết lại).** Viết lại toàn bộ `my_judge(cards)` bằng quay lui (float hoặc Fraction) không nhìn code.

**Bài 5 (mini challenge).** Viết `count_expressions(cards)` đếm **số chuỗi biểu thức khác nhau** (theo cách liệt kê của Cách 1, sau khi bỏ hoán vị trùng) có giá trị 24. Kiểm tra `[4, 1, 8, 7]` có ít nhất 1 và `[1, 2, 1, 2]` bằng 0.

In [25]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def combine_todo(a, b):
    # TODO: trả về set gồm a+b, a*b, a-b, b-a, a/b (nếu b != 0), b/a (nếu a != 0)
    pass

# Bài 4 — TODO
def my_judge(cards):
    # TODO: chuyển sang Fraction; dfs(nums): nếu 1 phần tử thì so với 24;
    #       ngược lại duyệt cặp i<j, rest, thử mọi giá trị combine, đệ quy
    pass

# Bài 5 — TODO
def count_expressions(cards):
    # TODO: duyệt set(permutations), product(OPS, repeat=3), 5 SHAPES; đếm số lần == 24
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: số lớn nhất tạo được từ bốn số 1 là bao nhiêu?
- Mức 2: `[2,2,2,3]`: thử `(2 + 2) * 2 * 3`.
- Mức 3: đáp án `False`, `True` (`(2+2)·2·3 = 24`), `True` (`(1+1+2)·6 = 24`).

**Bài 2.**
- Mức 1: thử tạo một phân số nhỏ từ 1, 3, 4.
- Mức 2: `3 / 4`, rồi `1 - 3/4 = 1/4`.
- Mức 3: `6 / (1/4) = 24`. Bảng: `[1,3,4,6] → [1,6,3/4] → [6,1/4] → [24]`.

**Bài 3.**
- Mức 1: bắt đầu bằng `out = {a + b, a * b}`.
- Mức 2: thêm hai phép trừ hai chiều.
- Mức 3: hai phép chia hai chiều, mỗi phép có điều kiện mẫu khác 0.

**Bài 4.**
- Mức 1: điểm dừng `len(nums) == 1`.
- Mức 2: `rest = [nums[t] for t in range(n) if t not in (i, j)]`.
- Mức 3: `if dfs(rest + [v]): return True`; cuối hàm `return False`.

**Bài 5.**
- Mức 1: dùng lại `SHAPES` và `apply` ở mục 5.
- Mức 2: không `return` sớm; tăng biến đếm.
- Mức 3: có thể so sánh với `judge_brute`: đếm > 0 ⇔ `True`.

## 19. Đáp án tham khảo

In [26]:
# Bài 1
assert Solution().judgePoint24([1, 1, 1, 1]) is False
assert Solution().judgePoint24([2, 2, 2, 3]) is True
assert Solution().judgePoint24([1, 1, 2, 6]) is True

# Bài 2
F = Fraction
step1 = F(3) / 4
step2 = 1 - step1
step3 = 6 / step2
assert (step1, step2, step3) == (F(3, 4), F(1, 4), 24)

# Bài 3
def combine_ans(a, b):
    out = {a + b, a * b, a - b, b - a}
    if b != 0:
        out.add(a / b)
    if a != 0:
        out.add(b / a)
    return out

assert combine_ans(F(3), F(8, 3)) == combine(F(3), F(8, 3))
assert F(1, 3) in combine_ans(F(3), F(8, 3))
print("Bài 1–3 OK")

Bài 1–3 OK


In [27]:
# Bài 4
def my_judge_ans(cards):
    def dfs(nums):
        if len(nums) == 1:
            return nums[0] == 24
        n = len(nums)
        for i in range(n):
            for j in range(i + 1, n):
                rest = [nums[t] for t in range(n) if t not in (i, j)]
                for v in combine_ans(nums[i], nums[j]):
                    if dfs(rest + [v]):
                        return True
        return False
    return dfs([Fraction(c) for c in cards])

for h in all_hands:
    assert my_judge_ans(h) == judge_backtrack_frac(h)

# Bài 5
def count_expressions_ans(cards):
    cnt = 0
    for p in set(permutations([Fraction(c) for c in cards])):
        for o1, o2, o3 in product(OPS, repeat=3):
            for shape in SHAPES:
                if shape(*p, o1, o2, o3) == 24:
                    cnt += 1
    return cnt

c1, c2 = count_expressions_ans([4, 1, 8, 7]), count_expressions_ans([1, 2, 1, 2])
print("số biểu thức = 24:", c1, c2)
assert c1 >= 1 and c2 == 0
random.seed(5)
for _ in range(15):
    h = [random.randint(1, 9) for _ in range(4)]
    assert (count_expressions_ans(h) > 0) == judge_brute(h)
print("Bài 4–5 OK")

số biểu thức = 24: 48 0
Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "chọn cặp, gộp, đệ quy":**

```python
def dfs(nums):
    if len(nums) == 1:
        return nums[0] == TARGET              # Fraction: so sánh chính xác
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            rest = [nums[t] for t in range(n) if t != i and t != j]
            for v in combine(nums[i], nums[j]):   # +, *, a-b, b-a, a/b, b/a
                if dfs(rest + [v]):
                    return True
    return False
```

**Template "tập giá trị của tập con":**

```python
vals(mask) = {x op y : sub ⊂ mask, x ∈ vals(sub), y ∈ vals(mask ^ sub)}
```

**Checklist trước khi submit:**

- [ ] Phép chia là chia **thực** (`/` hoặc `Fraction`), không phải `//`.
- [ ] So sánh với 24 bằng `Fraction ==` hoặc `abs(x - 24) < EPS`, **không** `float ==`.
- [ ] Đã kiểm tra mẫu khác 0 trước khi chia.
- [ ] Đã thử **cả hai chiều** cho `-` và `/` (hoặc duyệt cặp có thứ tự).
- [ ] Kết quả gộp được đưa **trở lại** danh sách để có thể ghép với bất kỳ số nào (sinh đủ mọi dạng ngoặc).
- [ ] Không dùng dấu trừ một ngôi, không ghép chữ số.
- [ ] Test `[4,1,8,7] → True`, `[1,2,1,2] → False`, `[3,3,8,8] → True`, `[1,1,1,1] → False`.
- [ ] Tên method đúng: `judgePoint24(self, cards)`.